# Lección 5.7 — Comparativa: ¿qué modelo uso?

> **Módulo 5 · Modelos clásicos supervisados** · Nivel 2 — Aprendiz
> ⏱️ Tiempo estimado: 110–130 min (lección con ejercicios intercalados ≈ 80 min · ejercicios finales ≈ 35 min)
> 📋 Requisitos previos: Lecciones 5.1–5.6, Lección 4.6 (validación cruzada), Lección 4.8 (elegir la métrica según el costo del error)

### 🎯 Objetivos
Al terminar esta lección podrás:
- Comparar los modelos del módulo en **forma de frontera, preparación de datos, velocidad, interpretabilidad y probabilidades**.
- Preparar datos **mixtos** (números y categorías) para cualquier modelo con un `ColumnTransformer` dentro de un pipeline.
- Organizar un **torneo justo** de modelos con `cross_val_predict`, sin tocar el conjunto de prueba.
- Elegir un modelo por lo que le importa al **negocio** (costo, capacidad, explicabilidad), no solo por la exactitud.
- Usar una tabla de decisión "¿qué modelo uso?" como punto de partida.

### 🧭 Cómo usar este notebook
| Icono | Significa |
|---|---|
| 📖 | Lee: explicación del concepto |
| ▶️ | Ejecuta la celda y observa el resultado |
| 🔮 | **Predice** antes de ejecutar: escribe tu apuesta y luego compruébala |
| ✋ | **Tu turno**: ejercicio. Las respuestas están ocultas en "👀 Ver respuesta" |
| ✏️ | Modifica un valor en el código y vuelve a ejecutar |

> 💡 No necesitas saber programar. El código está para que **veas** los conceptos en acción; solo tendrás que cambiar números.

## ▶️ Preparación
Ejecuta esta celda una vez (`Shift + Enter`). Carga las herramientas que usaremos.

In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.dummy import DummyClassifier
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split, cross_val_score, cross_val_predict
from sklearn.metrics import confusion_matrix, recall_score, precision_score
from sklearn.datasets import make_moons, make_circles, make_classification

plt.rcParams["figure.figsize"] = (8, 4.5)
print("✅ Todo listo")

---
## 1. 📖 La intuición: una caja de herramientas, no una navaja suiza

Un carpintero no pregunta *"¿cuál es la mejor herramienta?"*. Pregunta *"¿cuál es la mejor herramienta **para este trabajo**?"*. El martillo es perfecto para clavos y pésimo para tornillos.

Con los modelos pasa lo mismo. Ya lo viste en la Lección 1.6: **no hay almuerzo gratis**. Ningún modelo gana en todos los problemas. Lo que sí existe son **señales** que te dicen por dónde empezar:
- ¿Qué **forma** tiene la frontera entre clases?
- ¿Cuántos **datos** y cuántas **características** tienes?
- ¿Necesitas **explicar** cada decisión?
- ¿Necesitas **probabilidades** para decidir con costos?
- ¿Qué tan **rápido** tiene que entrenar y predecir?

### ✋ Ejercicio 1 — Calentamiento: primera intuición
Sin tablas todavía, solo con lo que recuerdas del módulo, ¿qué modelo probarías **primero** en cada caso?
1. Clasificar 300 correos en "queja" o "consulta", con muy pocos ejemplos etiquetados.
2. Un banco debe poder explicar **cada** rechazo al cliente con reglas simples.
3. Predecir el consumo eléctrico de una casa a partir de su tamaño, con una relación casi lineal, y explicar "cuánto suma cada m²".

✍️ *Mis respuestas:*

<details><summary><b>👀 Ver respuesta</b></summary>

1. **Naive Bayes** (Lección 5.6): texto + pocos datos es su terreno ideal.
2. **Árbol de decisión** poco profundo (Lección 5.4): se lee como reglas "si… entonces…". Una **regresión logística** también se puede explicar (con sus coeficientes), aunque no en forma de reglas.
3. **Regresión lineal** (Lección 5.1): relación casi lineal y un coeficiente que responde justo a "cuánto suma cada m²".

No hay respuestas únicas, pero sí **puntos de partida** razonables. El resto de la lección te da criterios para elegirlos.
</details>

---
## 2. 📖 La gran tabla comparativa

| | Regresión lineal / logística | KNN | Árbol de decisión | SVM | Naive Bayes |
|---|---|---|---|---|---|
| **Problemas** | Regresión / clasificación | Ambos | Ambos | Ambos | Clasificación |
| **Forma de la frontera** | Recta (salvo que crees características) | Cualquiera | "Escalones" (rectángulos) | Recta o cualquiera (kernel RBF) | Suave, según las campanas/frecuencias |
| **¿Hay que escalar?** | **Sí** (logística; y para comparar coeficientes) | **Sí, imprescindible** | No | **Sí, imprescindible** | No |
| **Categorías** | One-hot | One-hot (con cuidado) | One-hot u ordinal | One-hot | One-hot / `CategoricalNB` |
| **Interpretabilidad** | Alta (coeficientes) | Media (casos parecidos) | **Muy alta** (reglas) si es poco profundo | Baja (RBF) | Media (pistas por clase) |
| **Entrenar** | Rápido | Instantáneo | Rápido | **Lento** con muchas filas | **Rapidísimo** |
| **Predecir** | Rapidísimo | **Lento** con muchos datos | Rapidísimo | Medio | Rapidísimo |
| **Con pocos datos** | Bien | Regular | Regular (inestable) | Bien | **Muy bien** |
| **Muchas características** | Bien (con regularización) | **Mal** (dimensionalidad) | Bien | Bien | **Muy bien** (texto) |
| **Probabilidades** | **Bien calibradas** (logística) | A saltos (votos) | A saltos (hojas) | No directas (`decision_function`) | Exageradas |
| **¿Extrapola?** | Sí (sigue la recta) | No | No | Sí con kernel lineal; con RBF, poco | — |
| **Hiperparámetros clave** | `C` / `alpha` | `n_neighbors`, `weights` | `max_depth`, `min_samples_leaf` | `C`, `gamma`, `kernel` | `alpha`, `var_smoothing` |
| **Lección** | 5.1 · 5.2 | 5.3 | 5.4 | 5.5 | 5.6 |

### ✋ Ejercicio 2 — Consulta la tabla
1. Tienes 2 millones de filas y necesitas predecir en milisegundos en una app móvil. ¿Qué dos modelos descartarías primero?
2. Vas a usar la fórmula de umbral por costos de la Lección 5.2. ¿Qué modelo da las probabilidades más confiables para eso?
3. ¿Qué modelos puedes usar **sin** escalar?

✍️ *Mis respuestas:*

<details><summary><b>👀 Ver respuesta</b></summary>

1. **KNN** (predecir obliga a comparar con 2 millones de filas guardadas) y **SVM** (entrenar con 2 millones de filas es lentísimo).
2. La **regresión logística**: sus probabilidades suelen estar bien calibradas. Las de Naive Bayes son exageradas, y las de KNN y árboles van "a saltos".
3. **Árbol de decisión** y **Naive Bayes** (el gaussiano). Escalarlos no les hace daño (salvo `MultinomialNB`, que no admite negativos), pero tampoco les ayuda.
</details>

---
## 3. 📖 Cada modelo "ve" el mundo a su manera

La mejor forma de entender la tabla es **verla**. Vamos a entrenar los 5 clasificadores en 3 conjuntos de datos con formas distintas:
- **Lineal:** dos nubes separables (más o menos) con una recta.
- **Lunas:** dos medias lunas entrelazadas.
- **Círculos:** un círculo dentro de otro.

In [ ]:
X_lin, y_lin = make_classification(n_samples=300, n_features=2, n_redundant=0, n_informative=2,
                                   n_clusters_per_class=1, class_sep=1.2, random_state=2)
conjuntos = {
    "Lineal":   (X_lin, y_lin),
    "Lunas":    make_moons(n_samples=300, noise=0.25, random_state=0),
    "Círculos": make_circles(n_samples=300, noise=0.1, factor=0.45, random_state=1),
}
modelos = {
    "Logística":   make_pipeline(StandardScaler(), LogisticRegression()),
    "KNN (k=15)":  make_pipeline(StandardScaler(), KNeighborsClassifier(15)),
    "Árbol (d=4)": DecisionTreeClassifier(max_depth=4, random_state=0),
    "SVM (RBF)":   make_pipeline(StandardScaler(), SVC()),
    "Naive Bayes": GaussianNB(),
}

### 🔮 Ejercicio 3 — Antes de ver las fronteras
Usando lo que sabes de cada modelo, predice:
1. ¿Qué modelo fallará estrepitosamente en los **círculos**?
2. ¿Qué modelo dibujará fronteras en forma de **escalones**?
3. ¿Habrá algún modelo que funcione bien en los tres?

✍️ *Mis predicciones:*

In [ ]:
fig, axes = plt.subplots(len(conjuntos), len(modelos), figsize=(3.2 * len(modelos), 3 * len(conjuntos)))
tabla_cv = pd.DataFrame(index=list(conjuntos), columns=list(modelos), dtype=float)
for i, (nombre_d, (X, y)) in enumerate(conjuntos.items()):
    xx, yy = np.meshgrid(np.linspace(X[:, 0].min() - .5, X[:, 0].max() + .5, 200),
                         np.linspace(X[:, 1].min() - .5, X[:, 1].max() + .5, 200))
    for j, (nombre_m, modelo) in enumerate(modelos.items()):
        tabla_cv.loc[nombre_d, nombre_m] = cross_val_score(modelo, X, y, cv=5).mean()
        zz = modelo.fit(X, y).predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
        ax = axes[i, j]
        ax.contourf(xx, yy, zz, alpha=.3, cmap="coolwarm")
        ax.scatter(*X.T, c=y, cmap="coolwarm", edgecolor="k", s=8)
        ax.set_xticks([]); ax.set_yticks([])
        ax.set_title(f"{nombre_m}\nCV {tabla_cv.loc[nombre_d, nombre_m]:.0%}" if i == 0 else f"CV {tabla_cv.loc[nombre_d, nombre_m]:.0%}", fontsize=9)
    axes[i, 0].set_ylabel(nombre_d, fontsize=11)
plt.tight_layout(); plt.show()
tabla_cv.round(2)

<details><summary><b>👀 Ver respuesta</b></summary>

1. La **logística** se queda en ≈47 % en los círculos: su frontera es una recta.
2. El **árbol**: cada pregunta mira una sola característica, así que sus fronteras son líneas verticales y horizontales (aun así, con escalones, rodea bien los círculos: ≈98 %).
3. **KNN y SVM (RBF)** funcionan bien en los tres (≈95–100 %): pueden dibujar fronteras de cualquier forma. Naive Bayes resuelve los círculos (≈98 %, sus "campanas" de cada clase tienen distinto ancho) pero en las lunas se queda como la logística (≈83 %).

Y fíjate en la primera fila: en datos **lineales**, todos rondan el ≈95–96 %. Cuando el problema es simple, un modelo simple basta, y es más fácil de explicar.
</details>

---
## 4. 📖 Preparar datos reales: números **y** categorías

Los datos de verdad mezclan columnas numéricas (edad, monto) y categóricas (día de la semana, tipo de cita). Y cada modelo tiene sus necesidades:
- Las **categorías** se convierten en columnas 0/1 con **one-hot** (Lección 2.3): `OneHotEncoder`.
- Los **números** se escalan para logística, KNN y SVM: `StandardScaler`.

`ColumnTransformer` aplica **cada transformación a sus columnas**, y `make_column_selector` elige las columnas por tipo, para no escribirlas a mano:

```python
preprocesador = ColumnTransformer([
    ("num", StandardScaler(),                        make_column_selector(dtype_include="number")),
    ("cat", OneHotEncoder(handle_unknown="ignore"),  make_column_selector(dtype_include="category")),
])
modelo = make_pipeline(preprocesador, LogisticRegression())
```

- `handle_unknown="ignore"`: si en la validación o en producción aparece una categoría que no se vio al entrenar, pone todo en 0 en lugar de dar error.
- Al ir **dentro del pipeline**, el escalado y el one-hot se aprenden **solo con el entrenamiento** de cada pliegue: cero fuga de datos (Lección 2.6).

Vamos a usarlo en un caso: una red de clínicas quiere predecir qué pacientes **faltarán a su cita** para hacer sobrecupo.

In [ ]:
rng = np.random.default_rng(42)
n = 3_000
citas = pd.DataFrame({
    "edad":              rng.integers(18, 86, n),
    "dias_anticipacion": rng.integers(0, 61, n),
    "faltas_previas":    rng.poisson(0.5, n),
    "distancia_km":      rng.gamma(2, 5, n).round(1),
    "dia_semana":        rng.choice(["Lun", "Mar", "Mié", "Jue", "Vie"], n),
    "tipo_cita":         rng.choice(["general", "especialista", "laboratorio", "dental"], n),
    "recordatorio_sms":  rng.choice(["sí", "no"], n, p=[0.6, 0.4]),
    "seguro":            rng.choice(["público", "privado", "ninguno"], n, p=[0.5, 0.3, 0.2]),
})
z = (-2.6 + 0.035 * citas.dias_anticipacion + 0.6 * citas.faltas_previas + 0.03 * citas.distancia_km
     - 0.012 * (citas.edad - 40) + 0.5 * (citas.dia_semana == "Lun") + 0.4 * (citas.tipo_cita == "laboratorio")
     - 0.9 * (citas.recordatorio_sms == "sí") + 0.5 * (citas.seguro == "ninguno")
     + 1.0 * ((citas.edad < 30) & (citas.distancia_km > 15)))
citas["falta"] = (rng.random(n) < 1 / (1 + np.exp(-z))).astype(int)

# Las columnas de texto se marcan como "category" para que make_column_selector las encuentre
for col in ["dia_semana", "tipo_cita", "recordatorio_sms", "seguro"]:
    citas[col] = citas[col].astype("category")

X = citas.drop(columns="falta")
y = citas["falta"]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, stratify=y, random_state=0)
print(f"Faltan a su cita: {y.mean():.0%} · entrenamiento {len(X_tr)} · prueba {len(X_te)} (🔒 guardada hasta el final)")
print(X.dtypes.value_counts())
citas.head()

### 🔮 Ejercicio 4 — ¿Cuántas columnas?
Hay 4 columnas numéricas y 4 categóricas con 5, 4, 2 y 3 categorías. Después del preprocesador, ¿cuántas columnas tendrá la tabla que llega al modelo?

✍️ *Mi cálculo:*

In [ ]:
def nuevo_preprocesador():
    return ColumnTransformer([
        ("num", StandardScaler(), make_column_selector(dtype_include="number")),
        ("cat", OneHotEncoder(handle_unknown="ignore"), make_column_selector(dtype_include="category")),
    ])

pre = nuevo_preprocesador().fit(X_tr)
print("Columnas de salida:", len(pre.get_feature_names_out()))
print(list(pre.get_feature_names_out()))

<details><summary><b>👀 Ver respuesta</b></summary>

4 numéricas + (5 + 4 + 2 + 3) = **18 columnas**. Cada categoría se vuelve una columna 0/1: `cat__dia_semana_Lun`, `cat__seguro_ninguno`… Las numéricas conservan su nombre con el prefijo `num__`, ya escaladas.

> 💡 Con este preprocesador **todos** los modelos del módulo pueden recibir los mismos datos. Al árbol y a Naive Bayes el escalado no les hace falta, pero tampoco les estorba, y así comparamos en igualdad de condiciones.
</details>

---
## 5. 📖 Un torneo justo

Reglas de un torneo justo entre modelos:
1. **Mismos datos** y **misma preparación** (dentro de un pipeline).
2. **Validación cruzada** sobre el entrenamiento; el conjunto de prueba queda **guardado bajo llave** (Lección 4.6).
3. **La métrica que le importa al negocio**, no la que sea más cómoda (Lección 4.8).
4. Comparar contra una **línea base** "tonta" (Lección 4.7).

`cross_val_predict` (Lección 5.2) nos da, para cada paciente del entrenamiento, la predicción de un modelo que **no lo vio**. Con esas predicciones calculamos **todas** las métricas que queramos.

**La regla de negocio de la clínica:** si predecimos que alguien falta y hacemos sobrecupo, pero el paciente sí llega (**falso positivo**), hay espera y horas extra: cuesta **1**. Si no predecimos una falta (**falso negativo**), el consultorio queda vacío: cuesta **4**.

In [ ]:
COSTO_FN, COSTO_FP = 4, 1

def costo(y_real, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_real, y_pred, labels=[0, 1]).ravel()
    return (COSTO_FN * fn + COSTO_FP * fp) / len(y_real)

def evaluar(nombre, y_pred, y_real=y_tr):
    return {"modelo": nombre, "exactitud": np.mean(y_pred == y_real),
            "recall": recall_score(y_real, y_pred), "precisión": precision_score(y_real, y_pred, zero_division=0),
            "costo": costo(y_real, y_pred)}

candidatos = {
    "Logística":   LogisticRegression(max_iter=1000),
    "KNN (k=15)":  KNeighborsClassifier(n_neighbors=15),
    "Árbol (d=4)": DecisionTreeClassifier(max_depth=4, random_state=0),
    "SVM":         SVC(),
    "Naive Bayes": GaussianNB(),
}

### 🔮 Ejercicio 5 — ¿Quién gana?
Antes de ejecutar el torneo:
1. ¿Qué estrategia "tonta" cuesta menos: **nunca** hacer sobrecupo o hacerlo **siempre**? (Pista: faltan ≈26 %.)
2. ¿Crees que el modelo con **mejor exactitud** será también el de **menor costo**?

✍️ *Mis predicciones:*

In [ ]:
filas = [evaluar("Nunca sobrecupo", np.zeros(len(y_tr), dtype=int)),
         evaluar("Siempre sobrecupo", np.ones(len(y_tr), dtype=int))]
for nombre, modelo in candidatos.items():
    pred = cross_val_predict(make_pipeline(nuevo_preprocesador(), modelo), X_tr, y_tr, cv=5)
    filas.append(evaluar(nombre, pred))

torneo = pd.DataFrame(filas).set_index("modelo").round(3)
torneo.sort_values("costo")

<details><summary><b>👀 Ver respuesta</b></summary>

1. **Siempre sobrecupo** (≈0.74) le gana a **nunca** (≈1.03): con 4 a 1, dejar consultorios vacíos sale muy caro. Cuidado: "siempre" tiene una exactitud pésima (≈26 %) y aun así es la estrategia "tonta" más barata.
2. **No.** La logística tiene la mejor exactitud (≈0.77), pero con umbral 0.5 detecta a muy pocos de los que faltan (*recall* ≈0.28) y cuesta ≈0.79. **Naive Bayes**, con el *recall* más alto (≈0.35), es el modelo de menor costo (≈0.76), aunque su exactitud no es la mejor. Y lo más llamativo: **ningún modelo le gana a "siempre sobrecupo"**. Todos deciden con un umbral de 0.5, que supone que los dos errores cuestan lo mismo.
</details>

---
## 6. 📖 Enséñale al modelo lo que cuesta cada error

Tienes dos herramientas del módulo para que los modelos dejen de tratar igual los dos errores:
1. **`class_weight="balanced"`** (Lección 5.2): la clase minoritaria pesa más durante el entrenamiento. Disponible en logística, árbol y SVM (KNN y Naive Bayes no lo tienen).
2. **Mover el umbral** con `predict_proba` (Lección 5.2): umbral teórico $C_{FP}/(C_{FP}+C_{FN}) = 1/(1+4) = 0.2$.

### ✏️ Ejercicio 6 — Pesos balanceados
Ejecuta y compara con la tabla anterior. ¿Cuánto bajan los costos?

In [ ]:
filas_bal = []
for nombre, modelo in {"Logística balanceada": LogisticRegression(max_iter=1000, class_weight="balanced"),
                       "Árbol balanceado (d=4)": DecisionTreeClassifier(max_depth=4, class_weight="balanced", random_state=0),
                       "SVM balanceada": SVC(class_weight="balanced")}.items():
    pred = cross_val_predict(make_pipeline(nuevo_preprocesador(), modelo), X_tr, y_tr, cv=5)
    filas_bal.append(evaluar(nombre, pred))
pd.DataFrame(filas_bal).set_index("modelo").round(3)

<details><summary><b>👀 Ver respuesta</b></summary>

Los tres bajan mucho de costo (logística ≈0.58, SVM ≈0.59, árbol ≈0.66) y ahora sí le ganan a "siempre sobrecupo" (≈0.74). El precio: la exactitud baja (≈0.66–0.68) y la precisión también (≈0.39–0.43), porque hacen sobrecupo con más gente. Su *recall* sube de ≈0.15–0.28 a ≈0.59–0.67.
</details>

### 🔮 Ejercicio 7 — El umbral de la regla de negocio
Ahora la logística **sin pesos**, pero con el umbral teórico de 0.2. ¿Crees que quedará mejor o peor que la logística balanceada?

✍️ *Mi predicción:*

In [ ]:
UMBRAL = COSTO_FP / (COSTO_FP + COSTO_FN)       # ✏️ prueba otros valores
prob = cross_val_predict(make_pipeline(nuevo_preprocesador(), LogisticRegression(max_iter=1000)),
                         X_tr, y_tr, cv=5, method="predict_proba")[:, 1]

umbrales = np.arange(0.05, 0.61, 0.01)
plt.plot(umbrales, [costo(y_tr, (prob >= u).astype(int)) for u in umbrales])
plt.axvline(UMBRAL, ls="--", c="g", label=f"Umbral = {UMBRAL:.2f}")
plt.axhline(torneo.loc["Siempre sobrecupo", "costo"], ls=":", c="gray", label="Siempre sobrecupo")
plt.xlabel("Umbral"); plt.ylabel("Costo promedio"); plt.legend(); plt.show()
pd.DataFrame([evaluar(f"Logística, umbral {UMBRAL:.2f}", (prob >= UMBRAL).astype(int))]).set_index("modelo").round(3)

<details><summary><b>👀 Ver respuesta</b></summary>

Queda **empatada con la logística balanceada** (≈0.58) y le gana con claridad a "siempre sobrecupo" (≈0.74), con un *recall* aún mayor (≈0.76). La curva es casi plana entre 0.17 y 0.26 (costo ≈0.57–0.59): el 0.2 sale **de la regla de negocio** y los datos lo confirman.

Ventajas del umbral frente a `class_weight`: se cambia **sin reentrenar** (si mañana la regla pasa a 3 a 1, solo cambias el número) y es fácil de explicar: *"hacemos sobrecupo cuando el riesgo de falta supera 20 %"*.
</details>

---
## 7. 📖 Más allá de la métrica: velocidad y explicabilidad

Cuando varios modelos quedan casi empatados en costo, deciden otros criterios. Uno muy práctico es la **velocidad**. Medimos cuánto tarda cada modelo en entrenar con 10,000 citas y en predecir otras 10,000.

> 📝 Los tiempos exactos dependen de tu computadora; fíjate en el **orden de magnitud** y en quién es el más lento.

### 🔮 Ejercicio 8 — ¿Quién es el más lento?
1. ¿Qué modelo crees que tarda más en **entrenar**?
2. ¿Y en **predecir**?

✍️ *Mis predicciones:*

In [ ]:
X_grande, y_grande = make_classification(n_samples=20_000, n_features=18, random_state=0)
Xg_tr, Xg_te, yg_tr = X_grande[:10_000], X_grande[10_000:], y_grande[:10_000]

tiempos = []
for nombre, modelo in candidatos.items():
    pipe = make_pipeline(StandardScaler(), modelo)
    t0 = time.perf_counter(); pipe.fit(Xg_tr, yg_tr); t1 = time.perf_counter(); pipe.predict(Xg_te); t2 = time.perf_counter()
    tiempos.append({"modelo": nombre, "entrenar (s)": t1 - t0, "predecir (s)": t2 - t1})
pd.DataFrame(tiempos).set_index("modelo").round(3)

<details><summary><b>👀 Ver respuesta</b></summary>

1. **SVM**, con muchísima diferencia (segundos, frente a centésimas o décimas de segundo de los demás). Y con 10 veces más filas tardaría bastante más de 10 veces más.
2. **SVM y KNN** son los más lentos al predecir: la SVM compara cada caso con miles de vectores de soporte y KNN con las 10,000 filas guardadas. KNN es instantáneo al entrenar, pero paga al predecir. Logística, árbol y Naive Bayes predicen casi al instante.

En producción, esto puede decidir por sí solo: un modelo que tarda un segundo por predicción no sirve para una app que responde en milisegundos.
</details>

### ✋ Ejercicio 9 — Explicar una decisión
La gerente de la clínica pregunta: *"¿Por qué el modelo dice que este paciente va a faltar?"*. Mira la salida de la celda: un árbol balanceado de 3 niveles. Escribe **una regla** en lenguaje sencillo.

In [ ]:
arbol_exp = make_pipeline(nuevo_preprocesador(),
                          DecisionTreeClassifier(max_depth=3, class_weight="balanced", random_state=0)).fit(X_tr, y_tr)
nombres = [n.split("__")[1] for n in arbol_exp[0].get_feature_names_out()]
print(export_text(arbol_exp[-1], feature_names=nombres, class_names=["asiste", "falta"]))

<details><summary><b>👀 Ver respuesta</b></summary>

Como las numéricas están **escaladas**, los cortes salen en "desviaciones estándar" (p. ej. `dias_anticipacion <= 0.46`) y no en días. Primera lección práctica: **si vas a explicar un árbol, no lo escales**, para que los cortes queden en unidades reales.

Aun así se pueden leer las reglas. La primera pregunta es la **anticipación** con la que se agendó la cita (`dias_anticipacion`) y la segunda, si el paciente **no recibió recordatorio por SMS** (`recordatorio_sms_no`). Una regla posible: *"Los pacientes que agendaron **con mucha anticipación** y **no recibieron recordatorio SMS** tienden a faltar"* (las dos hojas de esa rama dicen "falta"). Otra: *"con poca anticipación, sin SMS y **sin seguro**, también tienden a faltar"*. Además, son **accionables**: mandar el SMS a todos, sobre todo a quien agenda con mucho tiempo.
</details>

---
## 8. 🧭 El diagrama "¿qué modelo uso?"

Úsalo como **punto de partida**, nunca como sentencia: la última palabra la tiene la validación cruzada con la métrica del negocio.

```
¿Qué quieres predecir?
 │
 ├── Un NÚMERO (regresión)
 │     ├── ¿Relación casi lineal o necesitas explicar el efecto de cada variable? ──► Regresión lineal (5.1)
 │     ├── ¿Vas a predecir fuera del rango visto? ──► Regresión lineal (los árboles y KNN no extrapolan)
 │     └── ¿Relación no lineal compleja? ──► Árbol (5.4) → mejor aún, ensambles (M6)
 │
 └── Una CATEGORÍA (clasificación)
       ├── ¿Es TEXTO o tienes MUY POCOS datos? ──► Naive Bayes (5.6) como línea base
       ├── ¿Debes EXPLICAR cada decisión con reglas? ──► Árbol poco profundo (5.4)
       ├── ¿Necesitas PROBABILIDADES fiables (umbral por costos)? ──► Regresión logística (5.2)
       ├── ¿Pocas características numéricas y datos moderados? ──► KNN escalado (5.3)
       ├── ¿Frontera compleja, hasta decenas de miles de filas, buscas precisión? ──► SVM RBF (5.5)
       └── ¿No sabes? ──► Empieza por la LOGÍSTICA (línea base sólida) y compara con un árbol y una SVM
```

## 9. 📖 Errores comunes al comparar modelos

| 🚨 Error | Por qué es un problema | Qué hacer |
|---|---|---|
| Elegir el modelo mirando el conjunto de prueba | El resultado final deja de ser honesto (fuga) | Comparar con validación cruzada; la prueba se usa **una vez** |
| Preparar los datos distinto para cada modelo, o fuera del pipeline | La comparación no es justa y puede haber fuga | Un mismo preprocesador dentro de un pipeline |
| Comparar solo por exactitud | Con clases desbalanceadas o errores de costo distinto, engaña | Usar la métrica del negocio (costo, *recall*, capacidad…) |
| Declarar ganador por diferencias de décimas | Puede ser ruido de la validación cruzada | Mirar la variación entre pliegues; ante empate, el más simple |
| Comparar modelos sin ajustar contra uno ajustado | Injusto para los demás | Ajustar todos con el mismo esfuerzo, o ninguno |
| Olvidar la línea base | Puedes celebrar un modelo que no le gana a "no hacer nada" | Incluir siempre las estrategias "tontas" |

### 🔮 Ejercicio 10 — ¿Diferencia real o ruido?
En el torneo, la SVM y Naive Bayes quedaron con exactitudes medias casi iguales (≈0.75). Antes de declarar ganador, mira **cada pliegue**. ¿Crees que la diferencia es real?

✍️ *Mi predicción:*

In [ ]:
for nombre in ["SVM", "Naive Bayes"]:
    puntajes = cross_val_score(make_pipeline(nuevo_preprocesador(), candidatos[nombre]), X_tr, y_tr, cv=5)
    print(f"{nombre:<10} pliegues: {puntajes.round(3)} → media {puntajes.mean():.3f} ± {puntajes.std():.3f}")

<details><summary><b>👀 Ver respuesta</b></summary>

Las medias se separan por **dos milésimas** (0.751 contra 0.749), mientras que de un pliegue a otro cada modelo varía mucho más (Naive Bayes va de 0.729 a 0.760; desviación ≈ ±0.006 y ±0.012). Además, cada uno gana en algunos pliegues. La diferencia es mucho menor que el ruido: **es un empate**. Ante un empate, elige el modelo **más simple y rápido** (aquí, Naive Bayes, que entrena y predice cientos de veces más rápido que la SVM).
</details>

---
## 10. 📋 Ficha resumen: los modelos del módulo en una línea

| Modelo | En una frase | Su mejor momento | Su talón de Aquiles |
|---|---|---|---|
| **Regresión lineal** (5.1) | Una recta: punto de partida + suma ponderada | Explicar efectos y extrapolar tendencias | Relaciones curvas, atípicos |
| **Regresión logística** (5.2) | La recta convertida en probabilidad | Línea base de clasificación; umbral por costos | Fronteras no lineales |
| **KNN** (5.3) | "Dime con quién andas" | Pocas características, explicar con casos parecidos | Sin escalar; muchas dimensiones; datos enormes |
| **Árbol de decisión** (5.4) | El juego de las 20 preguntas | Reglas explicables, sin preparar datos | Sobreajuste e inestabilidad |
| **SVM** (5.5) | La calle más ancha | Precisión con datos medianos y fronteras complejas | Lenta con muchos datos; sin probabilidades |
| **Naive Bayes** (5.6) | El detective que suma pistas | Texto y pocos datos; rapidez | Probabilidades exageradas |

## 11. 🧠 Resumen en 5 puntos
1. **No hay un modelo mejor en todo**: cada uno tiene una forma de frontera, unas necesidades de preparación y un "mejor momento".
2. Prepara los datos mixtos con un **`ColumnTransformer`** (escalado + one-hot) **dentro de un pipeline**: comparación justa y sin fuga.
3. Compara con **validación cruzada** (`cross_val_predict`) y la **métrica del negocio**, siempre frente a una **línea base**; la prueba se usa una sola vez.
4. El modelo de mayor exactitud **no** suele ser el más barato: `class_weight` y, sobre todo, **el umbral** convierten un buen modelo en una buena decisión.
5. Ante un empate, deciden la **simplicidad**, la **velocidad** y la **explicabilidad**.

---
## 12. 🏋️ Ejercicios finales

Ya practicaste cada idea por separado. Ahora toca **combinarlas**. Estos ejercicios son más largos (≈35 min).

### 12.1 Repaso rápido (opción múltiple)

**1.** ¿Qué modelos del módulo **necesitan** escalar las características numéricas?
- a) Árbol y Naive Bayes · b) Logística, KNN y SVM · c) Todos · d) Ninguno

**2.** ¿Para qué sirve `handle_unknown="ignore"` en `OneHotEncoder`?
- a) Para ignorar las columnas numéricas · b) Para no dar error si aparece una categoría que no se vio al entrenar · c) Para borrar las filas con faltantes · d) Para escalar

**3.** En validación cruzada, el modelo A tiene 0.81 ± 0.03 y el B 0.80 ± 0.03. B es un árbol de 3 niveles; A, una SVM RBF. El cliente exige explicaciones. ¿Qué eliges?
- a) A, porque tiene mayor media · b) B: la diferencia es ruido y B es explicable · c) Ninguno · d) Hay que mirar la prueba para desempatar

**4.** ¿Por qué el escalado va **dentro** del pipeline al hacer `cross_val_predict`?
- a) Por velocidad · b) Para que se aprenda solo con el entrenamiento de cada pliegue (sin fuga) · c) Porque `cross_val_predict` no acepta datos escalados · d) No importa

**5.** Tienes 5 millones de transacciones y necesitas una respuesta en menos de 10 ms. ¿Qué modelos son candidatos razonables?
- a) KNN y SVM RBF · b) Logística, árbol y Naive Bayes · c) Solo SVM · d) Solo KNN

✍️ *Mis respuestas:* 1 → _ · 2 → _ · 3 → _ · 4 → _ · 5 → _

<details><summary><b>👀 Ver respuestas</b></summary>

1. **b)** Se basan en distancias (KNN, SVM RBF) o en coeficientes y un optimizador (logística).
2. **b)** Evita errores con categorías nuevas en validación o producción.
3. **b)** Una décima de diferencia con ±0.03 de variación es un empate; manda la explicabilidad. d) es trampa.
4. **b)** Es la regla anti-fuga de la Lección 2.6.
5. **b)** Entrenan con muchos datos sin problema y predicen casi al instante.
</details>

### 12.2 Preguntas para pensar (sin ejecutar)

**A.** Para cada caso, propón un modelo de partida y **una** razón. (No hay una única respuesta.)
1. Una app que clasifica 50,000 tuits por día en 6 temas.
2. Un hospital quiere estimar los días de hospitalización de un paciente y explicarle al director el efecto de la edad.
3. Una aseguradora decide qué reclamos revisar por fraude; un fraude no detectado cuesta 30 veces más que una revisión.
4. Un laboratorio clasifica 400 muestras de suelo con 8 mediciones químicas.

**B. Caso de análisis 💼** En el torneo de la Sección 5, el modelo de menor costo con umbral 0.5 fue **Naive Bayes**. Un colega propone: *"Listo, usemos Naive Bayes"*. ¿Qué le responderías, considerando lo que viste en las secciones 6 y 7?

✍️ *Mis respuestas:*

<details><summary><b>👀 Ver respuestas</b></summary>

**A.** Algunas propuestas razonables:
1. **Naive Bayes** (o logística/SVM lineal): texto, muchas palabras como características y hay que ser rápido.
2. **Regresión lineal**: es regresión, y su coeficiente de edad responde justo a la pregunta del director.
3. **Regresión logística con umbral** $1/(1+30) ≈ 0.03$: necesitas probabilidades confiables para decidir con costos tan desiguales.
4. **KNN o SVM RBF escalados**: pocas filas, pocas características numéricas; compáralos con validación cruzada.

**B.** Que ganó **solo** porque todos decidían con umbral 0.5 y Naive Bayes, con sus probabilidades exageradas, marcaba a más gente. En cuanto le enseñamos el costo a los demás (pesos balanceados o umbral 0.2), la logística (≈0.58) y la SVM balanceada (≈0.59) lo **superaron con mucho** (Naive Bayes: ≈0.76, ni siquiera le ganaba a "siempre sobrecupo"). Además, la logística da probabilidades confiables y el umbral se explica con la regla de negocio. La comparación justa es **cada modelo en su mejor versión**, no todos con el umbral por defecto.
</details>

### 12.3 🧩 Práctica integradora: "Solo podemos hablar con 1 de cada 10"

Una empresa con 4,000 empleados quiere reducir las renuncias. El equipo de Recursos Humanos tiene **capacidad para entrevistar solo al 10 %** de la plantilla cada trimestre. La pregunta ya no es "¿cuál es el umbral?" sino **"si entrevistamos al 10 % con mayor riesgo según el modelo, ¿cuántos de ellos de verdad iban a renunciar?"**. A esta métrica se le llama **precisión en el top 10 %**.

Lo harás en 6 pasos cortos. Cada uno tiene su respuesta: **no avances sin revisarla**.

**Paso 1 ▶️ — Mira los datos**

In [ ]:
rng_rh = np.random.default_rng(8)
n_rh = 4_000
rh = pd.DataFrame({
    "antiguedad_anios":  rng_rh.integers(0, 25, n_rh),
    "salario_miles":     rng_rh.normal(25, 8, n_rh).clip(8).round(1),
    "horas_extra_mes":   rng_rh.poisson(8, n_rh),
    "satisfaccion":      rng_rh.integers(1, 6, n_rh),                 # encuesta 1-5
    "distancia_km":      rng_rh.gamma(2, 6, n_rh).round(1),
    "area":              rng_rh.choice(["ventas", "operaciones", "TI", "finanzas", "RR.HH."], n_rh),
    "modalidad":         rng_rh.choice(["presencial", "híbrido", "remoto"], n_rh, p=[0.5, 0.35, 0.15]),
    "ascenso_ult_2_anios": rng_rh.choice(["sí", "no"], n_rh, p=[0.25, 0.75]),
})
z_rh = (-1.8 - 0.08 * rh.antiguedad_anios - 0.05 * (rh.salario_miles - 25) + 0.06 * rh.horas_extra_mes
        - 0.45 * (rh.satisfaccion - 3) + 0.02 * rh.distancia_km + 0.5 * (rh.area == "ventas")
        - 0.4 * (rh.modalidad == "remoto") + 0.5 * (rh.ascenso_ult_2_anios == "no")
        + 1.2 * ((rh.satisfaccion <= 2) & (rh.horas_extra_mes > 10)))
rh["renuncia"] = (rng_rh.random(n_rh) < 1 / (1 + np.exp(-z_rh))).astype(int)
for col in ["area", "modalidad", "ascenso_ult_2_anios"]:
    rh[col] = rh[col].astype("category")

X_rh, y_rh = rh.drop(columns="renuncia"), rh.renuncia
Xr_tr, Xr_te, yr_tr, yr_te = train_test_split(X_rh, y_rh, test_size=0.25, stratify=y_rh, random_state=0)
print(f"Renuncian: {y_rh.mean():.0%} · entrenamiento {len(Xr_tr)} · prueba {len(Xr_te)}")
rh.head()

**Paso 2 🔮 — La línea base.** Si RR.HH. eligiera al 10 % **al azar**, ¿qué precisión esperas en ese grupo?

✍️ *Mi apuesta:*

<details><summary><b>👀 Ver respuesta</b></summary>

La misma proporción de renuncias de toda la empresa (≈21 %, la que imprimió el Paso 1): elegir al azar no concentra a nadie. (En el Paso 3, una selección al azar concreta da ≈24 %: es la variación normal de sortear solo 300 personas.) Cualquier modelo debe superar claramente ese número.
</details>

**Paso 3 ▶️ — La métrica.** Esta función ordena a los empleados por su puntaje de riesgo y mide qué proporción del 10 % más alto renunció de verdad. Funciona con **cualquier** puntaje que ordene: probabilidades (`predict_proba`) o distancias (`decision_function` de la SVM).

In [ ]:
def precision_top(y_real, puntaje, fraccion=0.10):
    n_top = int(len(y_real) * fraccion)
    top = np.argsort(puntaje)[::-1][:n_top]          # los de mayor riesgo
    return np.asarray(y_real)[top].mean()

# Comprobación: un puntaje al azar debería dar ≈ la proporción de renuncias
print(f"Al azar: {precision_top(yr_tr, np.random.default_rng(0).random(len(yr_tr))):.1%}")

**Paso 4 ▶️ — El torneo por capacidad.** Con `cross_val_predict` obtenemos puntajes "honestos" de cada modelo. Para la SVM usamos `method="decision_function"`; para los demás, `predict_proba`.

In [ ]:
resultados_rh = []
for nombre, modelo in candidatos.items():
    metodo = "decision_function" if nombre == "SVM" else "predict_proba"
    puntaje = cross_val_predict(make_pipeline(nuevo_preprocesador(), modelo), Xr_tr, yr_tr, cv=5, method=metodo)
    if puntaje.ndim == 2:
        puntaje = puntaje[:, 1]
    pred_05 = cross_val_predict(make_pipeline(nuevo_preprocesador(), modelo), Xr_tr, yr_tr, cv=5)
    resultados_rh.append({"modelo": nombre, "exactitud (umbral por defecto)": np.mean(pred_05 == yr_tr),
                          "precisión top 10 %": precision_top(yr_tr, puntaje)})
pd.DataFrame(resultados_rh).set_index("modelo").round(3).sort_values("precisión top 10 %", ascending=False)

✍️ *¿Qué modelo concentra mejor a los que se van? ¿Coincide con el de mejor exactitud?*

<details><summary><b>👀 Ver respuesta</b></summary>

La **logística** y la **SVM** encabezan la tabla: en el 10 % de mayor riesgo, alrededor de **6 de cada 10** iban a renunciar, unas **3 veces** más que al azar. El **árbol de 4 niveles** queda por debajo: sus probabilidades van "a saltos" (una por hoja) y muchos empleados empatan con el mismo puntaje, así que ordena peor dentro del grupo de riesgo. Naive Bayes y KNN quedan en medio.

La exactitud con umbral por defecto casi no distingue a los modelos (todos ≈0.80) y **no responde la pregunta de RR.HH.**: ellos no necesitan un "sí/no", sino una **lista ordenada** por riesgo.
</details>

**Paso 5 ✋ — Elige y justifica.** Escribe qué modelo elegirías para RR.HH. y por qué, en dos frases. Ten en cuenta que los gerentes querrán saber **por qué** alguien está en la lista.

✍️ *Mi elección:*

<details><summary><b>👀 Ver respuesta</b></summary>

La **regresión logística**: empata arriba en la métrica de capacidad, es rápida y sus coeficientes explican qué empuja el riesgo (satisfacción, horas extra, ascensos…). La SVM ordena igual de bien, pero no se puede explicar fácilmente a un gerente. Ante un empate, gana lo explicable.
</details>

**Paso 6 ▶️ — Examen final, una sola vez.** Entrenamos la logística con todo el entrenamiento y medimos la precisión en el top 10 % de la prueba. También vemos qué variables pesan más.

In [ ]:
final_rh = make_pipeline(nuevo_preprocesador(), LogisticRegression(max_iter=1000)).fit(Xr_tr, yr_tr)
puntaje_te = final_rh.predict_proba(Xr_te)[:, 1]
print(f"Precisión en el top 10 % (prueba): {precision_top(yr_te, puntaje_te):.1%}   vs. al azar: {yr_te.mean():.1%}")

coef = pd.Series(final_rh[-1].coef_[0], index=[n.split("__")[1] for n in final_rh[0].get_feature_names_out()])
print("\nLo que más AUMENTA el riesgo :", list(coef.sort_values(ascending=False).index[:4]))
print("Lo que más lo REDUCE        :", list(coef.sort_values().index[:4]))

✋ Escribe el mensaje para la directora de RR.HH. (3 líneas, sin tecnicismos).

<details><summary><b>👀 Ver un ejemplo</b></summary>

En la prueba, el top 10 % del modelo tiene ≈54 % de renuncias contra ≈21 % al azar: unas **2.5 veces** más. Es algo menos que el ≈62 % de la validación cruzada (en la prueba el top 10 % son solo 100 personas, así que hay bastante variación), pero la mejora es clara. Los factores que más elevan el riesgo son trabajar en **ventas**, **no haber tenido ascenso** y las **horas extra**; los que más lo reducen, la **satisfacción** alta, la **antigüedad**, el **salario** y el trabajo **remoto**.

> *"Si entrevistamos cada trimestre al 10 % de empleados que el modelo marca como de mayor riesgo, alrededor de la mitad de las entrevistas serán con personas que de verdad estaban pensando en irse: unas 2.5 veces mejor que elegir al azar. Las señales principales son la baja satisfacción, las horas extra y la falta de ascensos, sobre todo en ventas. Proponemos empezar por ahí y medir cada trimestre cuántas renuncias logramos evitar."*
</details>

> 🏆 **¿Listo para el proyecto?** En el [Proyecto del Módulo 5](5.8-proyecto-del-modulo.md) harás un torneo como este con datos reales de créditos bancarios: preprocesador mixto, cinco modelos, costos, `class_weight`, umbral… y un informe para el comité.

---
## 📚 Para profundizar
- 📘 **Documentación:** scikit-learn, ejemplo *Classifier comparison* (la figura de la Sección 3 con más modelos) y guía *Choosing the right estimator* (el "mapa" oficial de scikit-learn).
- 📘 **Documentación:** scikit-learn, *Column Transformer with Mixed Types* (preparar datos mixtos con pipelines).
- 📖 **Lectura:** David Wolpert, "No Free Lunch Theorems" (la idea detrás de la Sección 1, Lección 1.6).

---
⬅️ **Anterior:** [5.6 Naive Bayes: probabilidad con supuestos ingenuos](5.6-naive-bayes.ipynb) · ➡️ **Siguiente:** [5.8 Proyecto del módulo: Comité de crédito 🏦](5.8-proyecto-del-modulo.md)